# Làm sạch giao dịch và tính RFM

Notebook thực hiện phần dữ liệu của người 1 sau bước khảo sát tại `data_audit.ipynb`.
Chạy lần lượt từ trên xuống. File gốc được giữ nguyên; các đầu ra được tạo lại.
Cần Python 3.11, Java 17 và thư viện trong `requirements.txt`.

## 1. Chuẩn bị đường dẫn và PySpark

Xác định thư mục dự án và khởi động Spark với múi giờ UTC.
Đặt `JAVA_HOME` đến Java 17 trước khi mở Jupyter.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.prepare_data import prepare_data, start_spark
from src.rfm import save_rfm
from scripts.check_rfm import check_rfm

spark = start_spark()

## 2. Làm sạch và lưu Parquet

Căn chỉnh dòng 6 trường theo mẫu đã quan sát, kiểm tra mã khách, mã đơn, ngày và giá.
Dòng không hợp lệ được tách kèm lý do; không tự điền mã khách hoặc xóa dòng giống nhau.
Bảng dưới hiển thị số dòng qua bước làm sạch.

In [2]:
clean, cleaning_report = prepare_data(
    spark,
    ROOT / "data/raw/kz.csv",
    ROOT / "data/interim/purchases_normalized.csv",
    ROOT / "data/processed",
    ROOT / "reports",
)
display(pd.DataFrame({
    "metric": ["raw_rows", "clean_rows", "quarantine_rows", "accepted_repaired_rows"],
    "value": [cleaning_report[key] for key in
              ["raw_rows", "clean_rows", "quarantine_rows", "accepted_repaired_rows"]],
}))
display(pd.DataFrame(
    cleaning_report["rows_by_primary_reason"].items(), columns=["primary_reason", "rows"]
))

,metric,value
0,raw_rows,2633521
1,clean_rows,600784
2,quarantine_rows,2032737
3,accepted_repaired_rows,37961


,primary_reason,rows
0,nonpositive_price,568
1,missing_customer_id,2030733
2,date_outside_scope,1436
3,accepted,600784


## 3. Xem dữ liệu sạch và chất lượng trước–sau

Quan sát mẫu giao dịch sau xử lý và tỷ lệ thiếu theo cột.
`before_validation` là dữ liệu đã căn chỉnh cấu trúc, trước khi tách dòng không hợp lệ.
Thiếu ngành hàng hoặc thương hiệu vẫn được giữ nếu đủ thông tin tính RFM.

In [3]:
display(pd.DataFrame([row.asDict() for row in clean.limit(5).collect()]))
display(pd.read_csv(ROOT / "reports/cleaning_quality.csv"))

,source_record,row_status,event_time,order_id,product_id,category_id,category_code,brand,price,customer_id
0,3,full_width,2020-04-24 21:37:43,2294444024058086220,2273948319057183658,2268105430162997728,electronics.audio.headphone,huawei,77.5200,1515915625447879434
1,4,full_width,2020-04-24 21:37:43,2294444024058086220,2273948319057183658,2268105430162997728,electronics.audio.headphone,huawei,77.5200,1515915625447879434
2,14,full_width,2020-04-27 21:57:22,2296628237930857206,1515966223509089660,2268105410021949476,computers.components.cpu,intel,91.4100,1515915625451131565
3,15,full_width,2020-04-27 21:57:22,2296628237930857206,1515966223509089660,2268105410021949476,computers.components.cpu,intel,91.4100,1515915625451131565
4,16,full_width,2020-04-27 21:57:22,2296628237930857206,1515966223509089660,2268105410021949476,computers.components.cpu,intel,91.4100,1515915625451131565


,stage,column,rows,missing_count,missing_pct
0,before_validation,event_time,2633521,0,0.0000
1,before_validation,order_id,2633521,0,0.0000
2,before_validation,product_id,2633521,0,0.0000
3,before_validation,category_id,2633521,431954,16.4021
4,before_validation,category_code,2633521,1044156,39.6487
5,before_validation,brand,2633521,544624,20.6805
6,before_validation,price,2633521,0,0.0000
7,before_validation,user_id,2633521,2030733,77.1109
8,after_validation,event_time,600784,0,0.0000
9,after_validation,order_id,600784,0,0.0000


## 4. Tính RFM và đối chiếu tổng giá trị

Mỗi khách có một dòng: R là số ngày từ lần mua cuối đến ngày chốt;
F là số đơn riêng biệt; M là tổng giá trên các dòng mua hợp lệ.
Ngày chốt bằng ngày sau lần mua hợp lệ cuối cùng, theo UTC.
Tổng M được đối chiếu với tổng giá dữ liệu sạch trước khi lưu.

In [4]:
try:
    rfm_report = save_rfm(clean, ROOT / "data/processed", ROOT / "reports")
    display(pd.DataFrame(rfm_report.items(), columns=["metric", "value"]))
finally:
    clean.unpersist()
    spark.stop()

rfm = pd.read_parquet(ROOT / "data/processed/customer_rfm.parquet")
display(rfm.head())

,metric,value
0,reference_date,2020-11-22
1,timezone,UTC
2,customers,234359
3,orders,402993
4,repeat_customers,74783
5,repeat_customer_pct,31.9096
6,monetary_value,118372961.6200
7,monetary_reconciles,True
8,money_unit,Source price units; not accounting revenue or ...


,customer_id,recency_days,frequency_orders,monetary_value,last_purchase_at
0,1515915625447879434,32,3,2841.2900,2020-10-21 19:40:23+00:00
1,1515915625451131565,209,1,274.2300,2020-04-27 14:57:22+00:00
2,1515915625442675260,134,4,2337.6100,2020-07-11 04:19:59+00:00
3,1515915625452188656,207,1,20.8100,2020-04-29 12:10:33+00:00
4,1515915625451952331,207,1,20.8100,2020-04-29 13:14:42+00:00


## 5. Đối chiếu mẫu từ CSV gốc

Tính lại R, F, M cho 3 khách trực tiếp từ CSV gốc, rồi so sánh với Parquet.
Bảng hiển thị giá trị tính lại, giá trị đã lưu và kết quả đối chiếu.

In [5]:
check_rfm()
display(pd.read_csv(ROOT / "reports/rfm_hand_check.csv", dtype={"customer_id": "string"}))

Manual RFM checks passed for 3 customers.


,customer_id,manual_R,saved_R,manual_F,saved_F,manual_M,saved_M,matched
0,1515915625514704541,4,4,178,178,39878.72,39878.72,True
1,1515915625451131565,209,209,1,1,274.23,274.23,True
2,1515915625441980762,127,127,4,4,374.79,374.79,True


## 6. Đọc bảng bàn giao

Người 2 nhận `customer_rfm.parquet` để làm baseline và K-Means.
Người 3 dùng dữ liệu sạch và báo cáo chất lượng để dựng dashboard.
Schema, kết quả và giới hạn được ghi tại `docs/person1_handoff.md`.